# 04 — CNN Transfer Learning (ResNet50)

**Question:** does fine-tuning a pretrained ResNet50 clear the ≥ 0.80 top-1 accuracy target?

**Method:** freeze backbone, retrain classifier head, then unfreeze last block for a low-LR sweep.  Standard aug pipeline (random crop, flip, colour jitter).

## Setup

In [ ]:
import torch, torch.nn as nn
from torch.utils.data import DataLoader, random_split
from torchvision import models, transforms, datasets
from pathlib import Path
import pandas as pd, numpy as np, json

SEED = 42; torch.manual_seed(SEED); np.random.seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DATA = Path('../data/raw/products')
ART  = Path('../ml/artifacts'); ART.mkdir(parents=True, exist_ok=True)
print('device:', DEVICE)

## Data pipeline

In [ ]:
train_tf = transforms.Compose([
    transforms.Resize(256), transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(), transforms.ColorJitter(0.2, 0.2, 0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])
eval_tf = transforms.Compose([
    transforms.Resize(256), transforms.CenterCrop(224), transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])
full = datasets.ImageFolder(DATA, transform=train_tf)
n_tr = int(0.8*len(full)); n_va = int(0.1*len(full)); n_te = len(full)-n_tr-n_va
tr, va, te = random_split(full, [n_tr, n_va, n_te], generator=torch.Generator().manual_seed(SEED))
dl_tr = DataLoader(tr, batch_size=32, shuffle=True, num_workers=2)
dl_va = DataLoader(va, batch_size=64, num_workers=2)
dl_te = DataLoader(te, batch_size=64, num_workers=2)
print(len(full.classes), 'classes')

## Model — frozen backbone + new head

In [ ]:
net = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
for p in net.parameters(): p.requires_grad = False
net.fc = nn.Linear(net.fc.in_features, len(full.classes))
net = net.to(DEVICE)
opt = torch.optim.AdamW(net.fc.parameters(), lr=3e-4)
crit = nn.CrossEntropyLoss(label_smoothing=0.05)

## Head-only training loop

In [ ]:
def run_epoch(loader, train=False):
    net.train(train)
    tot=0; correct=0; loss_sum=0
    for x,y in loader:
        x,y = x.to(DEVICE), y.to(DEVICE)
        with torch.set_grad_enabled(train):
            out = net(x); loss = crit(out, y)
            if train: opt.zero_grad(); loss.backward(); opt.step()
        loss_sum += loss.item()*x.size(0); tot += x.size(0); correct += (out.argmax(1)==y).sum().item()
    return loss_sum/tot, correct/tot

EPOCHS = 5
for e in range(EPOCHS):
    tl, ta = run_epoch(dl_tr, train=True)
    vl, va_acc = run_epoch(dl_va, train=False)
    print(f'ep{e} train {ta:.3f}  val {va_acc:.3f}')

## Unfreeze last block, fine-tune with low LR

In [ ]:
for p in net.layer4.parameters(): p.requires_grad = True
opt = torch.optim.AdamW([{'params': net.layer4.parameters(), 'lr': 1e-5},
                        {'params': net.fc.parameters(),      'lr': 1e-4}])
for e in range(3):
    tl, ta = run_epoch(dl_tr, train=True)
    vl, va_acc = run_epoch(dl_va, train=False)
    print(f'ft{e} train {ta:.3f}  val {va_acc:.3f}')

## Test-set evaluation

In [ ]:
tl, te_acc = run_epoch(dl_te, train=False)
print(f'TEST top-1 accuracy: {te_acc:.3f}')
metrics = {'test_top1_accuracy': te_acc, 'classes': full.classes, 'model': 'resnet50-ft'}
(ART/'product_clf_metrics.json').write_text(json.dumps(metrics, indent=2))

## Save artifact

In [ ]:
torch.save({'state_dict': net.state_dict(), 'classes': full.classes}, ART/'product_clf.pt')
print('saved', ART/'product_clf.pt')

## Findings

- **Test top-1:** {test_acc} (target ≥ 0.80)
- **Biggest confusion:** _(fill in — often laptop ↔ tablet)_
- **Latency:** _(one forward pass ~ Xms on {device})_